<a href="https://colab.research.google.com/github/IbrahimHanafy2222/Computer-Vision-Project/blob/main/02_evaluation_and_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CV552 Sign-Language Recognition — Evaluation & Architecture Comparison

Consumes the artifacts produced by `01_training_pipelines.ipynb` (models, SVM features, training histories) from Google Drive and computes the comparative metrics required by the project rubric.

## 1. Mount Drive & locate artifacts

In [ ]:
# ==========================================
# Mount Drive + locate artifacts written by 01_training_pipelines.ipynb
# ==========================================
import os, json, glob
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT   = '/content/drive/MyDrive/CV552_SignLanguage'
MODELS_DIR   = os.path.join(DRIVE_ROOT, 'models')
FEATURES_DIR = os.path.join(DRIVE_ROOT, 'features')
HISTORY_DIR  = os.path.join(DRIVE_ROOT, 'history')
RESULTS_DIR  = os.path.join(DRIVE_ROOT, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

print('Models   :', sorted(os.listdir(MODELS_DIR)))
print('Features :', sorted(os.listdir(FEATURES_DIR)))
print('History  :', sorted(os.listdir(HISTORY_DIR)))
print('Results  :', RESULTS_DIR)

## 2. Reload the Sign-MNIST test split

In [3]:
# ==========================================
# Reload the Sign-MNIST test split (identical preprocessing as training notebook)
# ==========================================
import os, numpy as np, pandas as pd
os.environ['KAGGLE_USERNAME'] = "abdullahashiry"
os.environ['KAGGLE_KEY']      = "KGAT_331632d901a6cb7a05431b55135bd8c2"
!kaggle datasets download -d datamunge/sign-language-mnist --unzip -q

test_path = 'sign_mnist_test/sign_mnist_test.csv'
if not os.path.exists(test_path):
    test_path = 'sign_mnist_test.csv'
test_df = pd.read_csv(test_path)

y_test = test_df['label'].values
x_test = test_df.drop('label', axis=1).values.reshape(-1, 28, 28)

alphabet_mapping = {0:'A',1:'B',2:'C',3:'D',4:'E',5:'F',6:'G',7:'H',8:'I',
                    10:'K',11:'L',12:'M',13:'N',14:'O',15:'P',16:'Q',17:'R',
                    18:'S',19:'T',20:'U',21:'V',22:'W',23:'X',24:'Y'}
target_names = [alphabet_mapping[i] for i in sorted(set(y_test))]
print('Test set:', x_test.shape, y_test.shape)


Dataset URL: https://www.kaggle.com/datasets/datamunge/sign-language-mnist
License(s): CC0-1.0
Test set: (7172, 28, 28) (7172,)


## 3. Load every trained model

In [15]:
import joblib, numpy as np, tensorflow as tf
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_preprocess

svm_model       = joblib.load(os.path.join(MODELS_DIR, 'svm_hog.joblib'))
svm_feats       = np.load(os.path.join(FEATURES_DIR, 'svm_hog_features.npz'))
x_test_features = svm_feats['x_test']

def fix_lambda_layers(model, preprocess_fn):
    """Patch Lambda layers by position: 1st=grayscale→RGB, 2nd=preprocess_input."""
    lambdas = [l for l in model.layers if type(l).__name__ == 'Lambda']
    if len(lambdas) >= 1:
        lambdas[0].function = tf.image.grayscale_to_rgb
    if len(lambdas) >= 2:
        lambdas[1].function = preprocess_fn

def load_keras(name, custom_objects=None, preprocess_fn=None):
    path  = os.path.join(MODELS_DIR, name)
    model = tf.keras.models.load_model(path, compile=False, safe_mode=False,
                                       custom_objects=custom_objects)
    if preprocess_fn is not None:
        fix_lambda_layers(model, preprocess_fn)
    return model

cnn_custom    = load_keras('custom_cnn.keras')
cnn_augmented = load_keras('cnn_augmented.keras')
mobilenet     = load_keras('mobilenetv2_head.keras',
                           custom_objects={'preprocess_input': mob_preprocess},
                           preprocess_fn=mob_preprocess)
effnet_head   = load_keras('efficientnetb0_head.keras',
                           custom_objects={'preprocess_input': eff_preprocess},
                           preprocess_fn=eff_preprocess)
effnet_ft     = load_keras('efficientnetb0_finetuned.keras',
                           custom_objects={'preprocess_input': eff_preprocess},
                           preprocess_fn=eff_preprocess)

print('All models loaded.')

All models loaded.


## 4. Comparative metrics table

In [ ]:
# ==========================================
# Comparative metrics: accuracy, precision, recall, F1, confusion matrices
# ==========================================
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, precision_recall_fscore_support)
import matplotlib.pyplot as plt
import seaborn as sns

x_test_dl  = x_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0
x_test_eff = x_test.reshape(-1, 28, 28, 1).astype('float32')  # EfficientNet preprocess is baked in

predictions = {
    'SVM + HOG':                   svm_model.predict(x_test_features),
    'Custom CNN':                  cnn_custom.predict(x_test_dl).argmax(axis=1),
    'CNN + Augmentation':          cnn_augmented.predict(x_test_dl).argmax(axis=1),
    'MobileNetV2 (head)':          mobilenet.predict(x_test_dl).argmax(axis=1),
    'EfficientNetB0 (head)':       effnet_head.predict(x_test_eff).argmax(axis=1),
    'EfficientNetB0 (fine-tuned)': effnet_ft.predict(x_test_eff).argmax(axis=1),
}

rows = []
for name, y_pred in predictions.items():
    p, r, f, _ = precision_recall_fscore_support(y_test, y_pred, average='macro', zero_division=0)
    rows.append({'model': name,
                 'accuracy':  accuracy_score(y_test, y_pred),
                 'precision': p, 'recall': r, 'f1_macro': f})

results_df = pd.DataFrame(rows).sort_values('accuracy', ascending=False)
print(results_df.to_string(index=False))

# Save metrics table
csv_path = os.path.join(RESULTS_DIR, 'metrics_table.csv')
results_df.to_csv(csv_path, index=False)
print(f'\nSaved → {csv_path}')

## 5. Confusion matrices & training curves

In [ ]:
# ==========================================
# Per-model confusion matrices and training curves — saved to Drive/results/
# ==========================================
for name, y_pred in predictions.items():
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(12, 9))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=target_names, yticklabels=target_names, ax=ax)
    ax.set_title(f'Confusion Matrix - {name}')
    ax.set_ylabel('True'); ax.set_xlabel('Predicted')
    slug = name.replace(' ', '_').replace('(', '').replace(')', '').replace('+', 'plus')
    fig.savefig(os.path.join(RESULTS_DIR, f'cm_{slug}.png'), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)

# Training curves
for hist_path in sorted(glob.glob(os.path.join(HISTORY_DIR, '*.json'))):
    with open(hist_path) as f:
        h = json.load(f)
    name = os.path.basename(hist_path).replace('.json', '')
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(h.get('accuracy', []),     label='train')
    axes[0].plot(h.get('val_accuracy', []), label='val')
    axes[0].set_title(f'{name} - accuracy'); axes[0].legend(); axes[0].grid(True)
    axes[1].plot(h.get('loss', []),     label='train')
    axes[1].plot(h.get('val_loss', []), label='val')
    axes[1].set_title(f'{name} - loss'); axes[1].legend(); axes[1].grid(True)
    fig.tight_layout()
    fig.savefig(os.path.join(RESULTS_DIR, f'curves_{name}.png'), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)

print('All figures saved to', RESULTS_DIR)

## 6. Single-image prediction — all models

In [ ]:
# ==========================================
# Pick one test image, show true label, run all models
# ==========================================
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from skimage.feature import hog

def extract_features_single(image_2d):
    """Match the HOG pipeline from 01_training_pipelines.ipynb exactly."""
    img_uint8 = image_2d.astype(np.uint8)
    blurred   = cv2.GaussianBlur(img_uint8, (3, 3), 0)
    edges     = cv2.Canny(blurred, threshold1=50, threshold2=150)
    features  = hog(blurred, orientations=9,
                    pixels_per_cell=(7, 7),
                    cells_per_block=(2, 2),
                    block_norm='L2-Hys')
    edge_density = np.sum(edges) / (255.0 * 28 * 28)
    return np.append(features, edge_density).reshape(1, -1)

# --- pick a sample (change idx to any value 0–7171) ---
idx = np.random.randint(0, len(x_test))
img          = x_test[idx]          # (28, 28) uint8
true_letter  = alphabet_mapping[y_test[idx]]

# --- prepare per-model inputs ---
img_dl   = img.reshape(1, 28, 28, 1).astype('float32') / 255.0
img_eff  = img.reshape(1, 28, 28, 1).astype('float32')
hog_feat = extract_features_single(img)

def top1_letter(out):
    pred_idx = int(out) if (np.isscalar(out) or out.shape == ()) else int(np.argmax(out))
    return alphabet_mapping[pred_idx]

model_preds = {
    'SVM + HOG':                   top1_letter(svm_model.predict(hog_feat)[0]),
    'Custom CNN':                  top1_letter(cnn_custom.predict(img_dl, verbose=0)[0]),
    'CNN + Augmentation':          top1_letter(cnn_augmented.predict(img_dl, verbose=0)[0]),
    'MobileNetV2 (head)':          top1_letter(mobilenet.predict(img_dl, verbose=0)[0]),
    'EfficientNetB0 (head)':       top1_letter(effnet_head.predict(img_eff, verbose=0)[0]),
    'EfficientNetB0 (fine-tuned)': top1_letter(effnet_ft.predict(img_eff, verbose=0)[0]),
}

# --- plot ---
n_models = len(model_preds)
fig, axes = plt.subplots(1, n_models + 1, figsize=(3 * (n_models + 1), 3.5))

axes[0].imshow(img, cmap='gray')
axes[0].set_title(f'True label\n"{true_letter}"', fontsize=13, fontweight='bold')
axes[0].axis('off')

for ax, (mname, pred) in zip(axes[1:], model_preds.items()):
    correct = pred == true_letter
    color   = '#2ecc71' if correct else '#e74c3c'
    ax.imshow(img, cmap='gray')
    ax.set_title(f'{mname}\nPred: "{pred}"', fontsize=9)
    ax.axis('off')
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor(color)
        spine.set_linewidth(4)

fig.legend(handles=[mpatches.Patch(color='#2ecc71', label='Correct'),
                    mpatches.Patch(color='#e74c3c', label='Wrong')],
           loc='lower center', ncol=2, fontsize=10, frameon=False)
fig.suptitle(f'Test sample #{idx}  |  True: "{true_letter}"', fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, f'sample_{idx}_true_{true_letter}.png'),
            dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Saved to', RESULTS_DIR)